# Challenge 2 — Generative Quantum Art (FRQI)

All patterns 1–7 solved with FRQI circuits.

**Encoding recap.** For an $R\times C$ image we use $k=\lceil\log_2(RC)\rceil$ position qubits + 1 color qubit. Pixel $(r,c)$ has address $i = r\cdot C + c$. With $H^{\otimes k}$ on the positions we get an equal superposition over every address; the color qubit then carries the pixel value.

Reconstruction uses the **ratio method**: for each address $i$ we estimate $P(\text{color}=1\mid i)$ from the counts.

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from utils_quantum import run_simulation

def reconstruct_from_frqi(counts, n_pos, shape, grayscale=False):
    n_pix = shape[0]*shape[1]
    total = np.zeros(n_pix); ones = np.zeros(n_pix)
    for bs, c in counts.items():
        bs = bs.replace(' ', '')
        idx = int(bs[1:], 2)
        if idx < n_pix:
            total[idx] += c
            if bs[0] == '1': ones[idx] += c
    img = np.zeros(n_pix)
    for i in range(n_pix):
        if total[i] > 0:
            r = ones[i]/total[i]
            img[i] = r*255 if grayscale else (255 if r > 0.5 else 0)
    return img.reshape(shape)

def _apply_x_mask(qc, bits_msb, pos):
    k = len(pos)
    for j, b in enumerate(bits_msb):
        if b == '0': qc.x(pos[k-1-j])

def show(img, title, cmap='gray'):
    plt.figure(figsize=(3,3))
    plt.imshow(img, cmap=cmap, vmin=0, vmax=255, interpolation='nearest')
    plt.title(title); plt.axis('off'); plt.show()

def n_pos_for(shape):
    return max(1, math.ceil(math.log2(shape[0]*shape[1])))

## Pattern 1 — Random Noise

Put **both** the address register *and* the color qubit in $|+\rangle$. Each shot then yields a uniformly random `(address, color)` pair — an independent 50/50 coin per pixel.

In [ ]:
def pattern1(shape=(8,8)):
    n = n_pos_for(shape)
    qc = QuantumCircuit(n+1)
    qc.h(range(n)); qc.h(n)
    qc.measure_all()
    return qc, n

qc, n = pattern1()
counts = run_simulation(qc, shots=200*2**n)
show(reconstruct_from_frqi(counts, n, (8,8)), 'Random noise (sample)')

## Pattern 2 — Checkerboard (+ negative)

Color = `LSB(row) XOR LSB(col)`. For $C=2^b$ columns, bit 0 of $i$ is `LSB(col)` and bit $b$ of $i$ is `LSB(row)`. Two CNOTs onto the color qubit do the XOR. Append `X` on the color qubit for the negative.

In [ ]:
def pattern2(shape=(8,8), negative=False):
    R,C = shape; b = int(math.log2(C)); a = int(math.log2(R)); n = a+b
    qc = QuantumCircuit(n+1)
    qc.h(range(n))
    qc.cx(0, n)
    qc.cx(b, n)
    if negative: qc.x(n)
    qc.measure_all()
    return qc, n

for neg in (False, True):
    qc, n = pattern2(negative=neg)
    show(reconstruct_from_frqi(run_simulation(qc, shots=100*2**n), n, (8,8)),
         'Negative' if neg else 'Checkerboard')

## Pattern 3 — Horizontal lines

Color depends on `LSB(row)`: one CNOT from the row-LSB qubit to the color qubit.

In [ ]:
def pattern3(shape=(8,8)):
    R,C = shape; b = int(math.log2(C)); a = int(math.log2(R)); n = a+b
    qc = QuantumCircuit(n+1); qc.h(range(n)); qc.cx(b, n); qc.measure_all()
    return qc, n
qc, n = pattern3()
show(reconstruct_from_frqi(run_simulation(qc, shots=100*2**n), n, (8,8)), 'Horizontal lines')

## Pattern 4 — Vertical lines

Same idea using `LSB(col)` (qubit 0).

In [ ]:
def pattern4(shape=(8,8)):
    R,C = shape; b = int(math.log2(C)); a = int(math.log2(R)); n = a+b
    qc = QuantumCircuit(n+1); qc.h(range(n)); qc.cx(0, n); qc.measure_all()
    return qc, n
qc, n = pattern4()
show(reconstruct_from_frqi(run_simulation(qc, shots=100*2**n), n, (8,8)), 'Vertical lines')

## Pattern 5 — Nested squares (per-pixel MCX)

Layer $L(r,c)=\min(r,c,R-1-r,C-1-c)$. White when $L$ is odd. For each such pixel, sandwich an `MCX` between `X`-masks so the controls fire exactly on that address.

In [ ]:
def pattern5(shape=(8,8)):
    R,C = shape; n = n_pos_for(shape)
    qc = QuantumCircuit(n+1); pos=list(range(n)); col=n
    qc.h(pos)
    for r in range(R):
        for c in range(C):
            L = min(r,c,R-1-r,C-1-c)
            if L % 2 == 0: continue
            addr = format(r*C+c, f'0{n}b')
            _apply_x_mask(qc, addr, pos)
            qc.mcx(pos, col)
            _apply_x_mask(qc, addr, pos)
    qc.measure_all(); return qc, n
qc, n = pattern5()
show(reconstruct_from_frqi(run_simulation(qc, shots=200*2**n), n, (8,8)), 'Nested squares')

## Pattern 6 — Grayscale nested squares

Same addressing, but the color qubit is rotated with `mcry(2θ, …)` where $\theta=\arcsin\sqrt{I}$ encodes intensity $I\in[0,1]$ as $P(|1\rangle)=I$.

In [ ]:
def pattern6(shape=(8,8)):
    R,C = shape; n = n_pos_for(shape)
    qc = QuantumCircuit(n+1); pos=list(range(n)); col=n
    qc.h(pos)
    max_L = min(R,C)//2 - 1
    for r in range(R):
        for c in range(C):
            L = min(r,c,R-1-r,C-1-c)
            I = L / max(1, max_L)
            theta = math.asin(math.sqrt(I))
            if theta == 0: continue
            addr = format(r*C+c, f'0{n}b')
            _apply_x_mask(qc, addr, pos)
            qc.mcry(2*theta, pos, col)
            _apply_x_mask(qc, addr, pos)
    qc.measure_all(); return qc, n
qc, n = pattern6()
show(reconstruct_from_frqi(run_simulation(qc, shots=400*2**n), n, (8,8), grayscale=True), 'Grayscale nested')

## Pattern 7 — Sierpinski + inverse (bonus)

Add a **selection qubit** $s$ in $|+\rangle$. For each pixel, apply an `MCX` controlled on the address qubits **and** on $s$:

- mask `=1` → control $s$ on `0` (white when $s=0$)
- mask `=0` → control $s$ on `1` (white when $s=1$ — inverted)

Split the counts by the measured value of $s$ to obtain the two reconstructions.

In [ ]:
def sierpinski_mask(size):
    m = np.zeros((size,size), dtype=int)
    for r in range(size):
        for c in range(size):
            if c <= r and (r & c) == c: m[r,c] = 1
    return m

def pattern7(size=8):
    shape = (size,size); n = n_pos_for(shape); mask = sierpinski_mask(size)
    qc = QuantumCircuit(n+2); pos=list(range(n)); s=n; col=n+1
    qc.h(pos); qc.h(s)
    for r in range(size):
        for c in range(size):
            addr = format(r*size+c, f'0{n}b')
            _apply_x_mask(qc, addr, pos)
            if mask[r,c] == 1: qc.x(s)
            qc.mcx(pos+[s], col)
            if mask[r,c] == 1: qc.x(s)
            _apply_x_mask(qc, addr, pos)
    qc.measure_all()
    counts = run_simulation(qc, shots=400*2**(n+1))
    grp = {0:{},1:{}}
    for bs, c in counts.items():
        bs = bs.replace(' ', '')
        s_b = int(bs[1])
        key = bs[0] + bs[2:]
        grp[s_b][key] = grp[s_b].get(key, 0) + c
    return reconstruct_from_frqi(grp[0], n, shape), reconstruct_from_frqi(grp[1], n, shape), mask*255

a, b, m = pattern7(8)
fig, ax = plt.subplots(1,3, figsize=(9,3))
for x, im, t in zip(ax, [m, a, b], ['classical mask','s=0 (mask)','s=1 (inverse)']):
    x.imshow(im, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    x.set_title(t); x.axis('off')
plt.tight_layout(); plt.show()